# LMA Phase 3 — Reasoning Finetuning, Evaluation & Attention Comparison

Finetunes both Phase 2 pretrained models (Model H = Hindi, Model L = Nepali) on the
Phase 3 synthetic comparative-reasoning dataset (`report/phase3_reasoning_report.md`
§1), independently per language, on **GPU T4 x2** (one model per GPU) — much shorter
than Phase 2's pretraining runs, since the reasoning dataset is a few thousand short
examples rather than ~930M tokens.

After finetuning, this notebook also runs:
- **Reasoning evaluation** (`reasoning_eval.py`): exact-match accuracy on the reasoning
  test set, for both the *pretrained* (zero-shot baseline) and the *finetuned*
  checkpoint, both languages.
- **Attention comparison** (`attention_analysis.py --prompts-file`): heatmaps/entropy/
  distance for the *same* fixed comparative-reasoning prompts, run once against the
  pretrained checkpoint and once against the finetuned checkpoint, so the two are
  directly comparable.

**Before running:**

1. **Add these input datasets** (Add Input):

   | Dataset | Contents |
   | --- | --- |
   | `sohailmemon/lma-individual-files` | (re-versioned) `hindi/tokenizer/`, `nepali/tokenizer/`, `scripts/` (incl. `scripts/training/finetune.py`, `scripts/reasoning/`), `hindi/reasoning/`, `nepali/reasoning/` |
   | `sohailmemon/lma-secrets` | private dataset holding `secrets.json` with `HF_TOKEN`/`WANDB_API_KEY` (this notebook is driven via the Kaggle API, so the usual Kaggle-Secrets-UI approach — see `kaggle_pretrain.ipynb` — doesn't carry over on an API push; see `Kaggle/kaggle-cli#582`) |

2. **Set accelerator**: Notebook settings -> Accelerator -> **GPU T4 x2**.
3. **Enable internet**: Notebook settings -> Internet -> on (needed for the HF Hub /
   W&B pushes).


## 0. Setup

In [ ]:
!pip install -q sentencepiece wandb huggingface_hub


In [ ]:
import shutil
from pathlib import Path

FILES_DS = Path("/kaggle/input/lma-individual-files")
SECRETS_DS = Path("/kaggle/input/lma-secrets")
WORK = Path("/kaggle/working")

# scripts.* does package-relative imports, so "scripts" needs to live at the top of a
# writable, importable location - copy it out of the read-only /kaggle/input mount.
if (WORK / "scripts").exists():
    shutil.rmtree(WORK / "scripts")
shutil.copytree(FILES_DS / "scripts", WORK / "scripts")

for lang in ("hindi", "nepali"):
    dest = WORK / lang
    dest.mkdir(exist_ok=True)
    for sub in ("tokenizer", "reasoning"):
        if (dest / sub).exists():
            shutil.rmtree(dest / sub)
        shutil.copytree(FILES_DS / lang / sub, dest / sub)

import sys
sys.path.insert(0, str(WORK))
print("scripts + per-language tokenizer/reasoning data copied under", WORK)


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print("gpu count:", n_gpu)
for i in range(n_gpu):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i))

assert n_gpu >= 2, (
    "this notebook pins Hindi to cuda:0 and Nepali to cuda:1 - set the accelerator to "
    "GPU T4 x2 in notebook settings."
)


### Secrets

Reads `HF_TOKEN`/`WANDB_API_KEY` from the mounted `lma-secrets` private dataset
(`secrets.json`) instead of Kaggle's interactive Secrets UI - see the notebook intro
for why.


In [ ]:
import json
import os

secrets = json.loads((SECRETS_DS / "secrets.json").read_text())
os.environ["HF_TOKEN"] = secrets["HF_TOKEN"]
os.environ["WANDB_API_KEY"] = secrets["WANDB_API_KEY"]
print("loaded HF_TOKEN and WANDB_API_KEY from", SECRETS_DS / "secrets.json")

assert os.environ.get("HF_TOKEN"), "secrets.json is missing HF_TOKEN"
assert os.environ.get("WANDB_API_KEY"), "secrets.json is missing WANDB_API_KEY"


## 1. Config

`PRETRAIN_HF_REPOS` are Phase 2's already-public checkpoints (source for the finetune
starting point *and* the "pretrained" zero-shot baseline in evaluation). `HF_REPOS` are
the new Phase 3 finetuned-checkpoint repos, created (public) on first run.


In [ ]:
LANGS = ["hindi", "nepali"]

EPOCHS = 8
BATCH_SIZE = 32
WARMUP_ITERS = 50
LR = 3e-5
MIN_LR = 3e-6
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
LOG_INTERVAL = 20
EVAL_INTERVAL = 50   # full val pass + best.pt check every 50 iters, not just once per
                     # epoch - a local dry run found the true best val loss at iter
                     # ~100, well before even one epoch (500 iters at this dataset size)

WANDB_PROJECT = "lma-individual-reasoning-finetune"

HF_USERNAME = "sohailtsm"
PRETRAIN_HF_REPOS = {lang: f"{HF_USERNAME}/lma-{lang}-gpt" for lang in LANGS}
HF_REPOS = {lang: f"{HF_USERNAME}/lma-{lang}-gpt-reasoning" for lang in LANGS}
HF_PUBLIC = True

# Auto-filled below by checking HF_REPOS for an existing last.pt (resume an interrupted
# finetune run). Leave as None here.
RESUME = {"hindi": None, "nepali": None}


### Pull each language's Phase 2 pretrained checkpoint, and check for a Phase 3
resume point

Same auto-resume pattern as `kaggle_pretrain.ipynb`: if `HF_REPOS[lang]` (the
*finetuned* repo) already has a `last.pt`, resume the finetune from there; otherwise
start finetuning fresh from the Phase 2 pretrained `best.pt`.


In [ ]:
from huggingface_hub import hf_hub_download

INIT_CHECKPOINT = {}
for lang in LANGS:
    INIT_CHECKPOINT[lang] = hf_hub_download(PRETRAIN_HF_REPOS[lang], "best.pt")
    print(f"[{lang}] pretrained checkpoint: {INIT_CHECKPOINT[lang]}")

for lang in LANGS:
    if RESUME[lang] is False:
        continue
    try:
        RESUME[lang] = hf_hub_download(HF_REPOS[lang], "last.pt")
        print(f"[{lang}] found existing finetune checkpoint on {HF_REPOS[lang]}, will resume from it")
    except Exception as e:
        print(f"[{lang}] no existing finetune checkpoint on {HF_REPOS[lang]} ({type(e).__name__}); "
              f"starting from the pretrained checkpoint")


## 2. Finetune both models in parallel (GPU T4 x2)

Hindi on `cuda:0`, Nepali on `cuda:1`, as two subprocesses - same streaming-log pattern
as `kaggle_pretrain.ipynb`.


In [ ]:
import os
import subprocess
import time


def finetune_async(lang: str, device: str, log_path) -> subprocess.Popen:
    out_dir = WORK / lang / "reasoning_checkpoints"
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "-u", "-m", "scripts.training.finetune",
        "--lang", lang,
        "--data-dir", str(WORK / lang / "reasoning" / "data"),
        "--tokenizer", str(WORK / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
        "--out-dir", str(out_dir),
        "--device", device,
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--warmup-iters", str(WARMUP_ITERS),
        "--lr", str(LR),
        "--min-lr", str(MIN_LR),
        "--weight-decay", str(WEIGHT_DECAY),
        "--grad-clip", str(GRAD_CLIP),
        "--log-interval", str(LOG_INTERVAL),
        "--eval-interval", str(EVAL_INTERVAL),
        "--wandb-project", WANDB_PROJECT,
        "--wandb-run-name", f"{lang}-gpt-reasoning",
        "--hf-repo-id", HF_REPOS[lang],
    ]
    if HF_PUBLIC:
        cmd.append("--hf-public")
    if RESUME.get(lang):
        cmd += ["--resume", RESUME[lang]]
    else:
        cmd += ["--init-checkpoint", INIT_CHECKPOINT[lang]]
    log_f = open(log_path, "w")
    return subprocess.Popen(cmd, cwd=WORK, stdout=log_f, stderr=subprocess.STDOUT)


def stream_until_done(jobs: dict, poll_s: float = 5.0) -> None:
    positions = {lang: 0 for lang in jobs}
    while any(p.poll() is None for p, _ in jobs.values()):
        for lang, (p, log_path) in jobs.items():
            with open(log_path) as f:
                f.seek(positions[lang])
                new_text = f.read()
                positions[lang] = f.tell()
            for line in new_text.splitlines():
                print(f"[{lang}] {line}")
        time.sleep(poll_s)
    for lang, (p, log_path) in jobs.items():
        with open(log_path) as f:
            f.seek(positions[lang])
            new_text = f.read()
        for line in new_text.splitlines():
            print(f"[{lang}] {line}")
        print(f"[{lang}] process exited with code {p.returncode}")


In [ ]:
jobs = {
    "hindi": (finetune_async("hindi", "cuda:0", WORK / "hindi_finetune.log"), WORK / "hindi_finetune.log"),
    "nepali": (finetune_async("nepali", "cuda:1", WORK / "nepali_finetune.log"), WORK / "nepali_finetune.log"),
}
stream_until_done(jobs)

for lang, (p, _) in jobs.items():
    if p.returncode != 0:
        raise RuntimeError(f"{lang} finetuning failed with exit code {p.returncode} - see "
                            f"{WORK / (lang + '_finetune.log')}")


## 3. Reasoning evaluation: pretrained vs. finetuned

Runs `reasoning_eval.py` against both the Phase 2 pretrained checkpoint (zero-shot
baseline) and the just-finetuned checkpoint, for both languages - 4 runs, each result
merged into one `<lang>_reasoning_metrics.json` per language (keyed by label).


In [ ]:
FINETUNED_CHECKPOINT = {lang: WORK / lang / "reasoning_checkpoints" / "best.pt" for lang in LANGS}

for lang in LANGS:
    out_dir = WORK / lang / "reasoning" / "eval"
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / f"{lang}_reasoning_metrics.json"

    for label, ckpt in (("pretrained", INIT_CHECKPOINT[lang]), ("finetuned", FINETUNED_CHECKPOINT[lang])):
        cmd = [
            "python", "-u", "-m", "scripts.evaluation.reasoning_eval",
            "--lang", lang, "--label", label,
            "--checkpoint", str(ckpt),
            "--tokenizer", str(WORK / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
            "--test-jsonl", str(WORK / lang / "reasoning" / "data" / "test.jsonl"),
            "--out", str(metrics_path),
        ]
        print(f"=== {lang} / {label} ===")
        subprocess.run(cmd, cwd=WORK, check=True)


## 4. Attention comparison: pretrained vs. finetuned

Runs `attention_analysis.py` with `--prompts-file` (fixed comparative-reasoning
prompts, `report/phase3_reasoning_report.md` §1) against both checkpoints, so the
heatmaps/entropy/distance are on identical input and directly comparable.


In [ ]:
for lang in LANGS:
    prompts_file = WORK / lang / "reasoning" / "attention_prompts.txt"
    for label, ckpt in (("pretrained", INIT_CHECKPOINT[lang]), ("finetuned", FINETUNED_CHECKPOINT[lang])):
        out_dir = WORK / lang / "eval" / "attention_reasoning" / label
        cmd = [
            "python", "-u", "-m", "scripts.evaluation.attention_analysis",
            "--lang", lang,
            "--checkpoint", str(ckpt),
            "--tokenizer", str(WORK / lang / "tokenizer" / f"{lang}_unigram_16k.model"),
            "--prompts-file", str(prompts_file),
            "--n-sentences", "6",
            "--out-dir", str(out_dir),
        ]
        print(f"=== {lang} / {label} attention ===")
        subprocess.run(cmd, cwd=WORK, check=True)


## 5. Summary

- **Finetuned checkpoints**: `/kaggle/working/<lang>/reasoning_checkpoints/{best,last}.pt`,
  pushed to `https://huggingface.co/<HF_USERNAME>/lma-<lang>-gpt-reasoning` (public).
- **Reasoning metrics**: `/kaggle/working/<lang>/reasoning/eval/<lang>_reasoning_metrics.json`
  (pretrained vs. finetuned accuracy, by category/form, qualitative examples).
- **Attention comparison**: `/kaggle/working/<lang>/eval/attention_reasoning/{pretrained,finetuned}/`
  (heatmaps + `<lang>_attention_summary.json` each).
- **Weights & Biases**: `https://wandb.ai/<your-entity>/lma-individual-reasoning-finetune`.


In [ ]:
for lang in LANGS:
    print(f"--- {lang} ---")
    ckpt_dir = WORK / lang / "reasoning_checkpoints"
    for f in sorted(ckpt_dir.iterdir()):
        print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
    metrics = json.loads((WORK / lang / "reasoning" / "eval" / f"{lang}_reasoning_metrics.json").read_text())
    for label in ("pretrained", "finetuned"):
        print(f"  {label} accuracy: {metrics[label]['accuracy']:.4f}")
    print(f"  HF repo: https://huggingface.co/{HF_REPOS[lang]}")
